In [ ]:
import os
from tqdm import tqdm

import pandas as pd

from pmhclib.components.pmhc import TCRpMHCII
from pmhclib.modeling.scoring import DockQScorer

In [ ]:
template_df = pd.read_csv("../data/mhcii_tcr_templates.csv", index_col="pdb_id")
pdb_ids = template_df.index.to_list()

In [ ]:
ensembles_dir = "../data/mhcii_tcr_ensembles/"
sampling_methods = ["static/crystal", "static/pandora2", "ensemble/annealing", "ensemble/pandora2"]
docking_method = "haddock3_rigidbody"

In [ ]:
df = None

for pdb_id in pdb_ids:
    for sampling_method in sampling_methods:
        ensemble_dir = os.path.join(pdb_id, sampling_method, docking_method)
        csv_fp = os.path.join(ensembles_dir, ensemble_dir, "stats.csv")
        if not os.path.exists(csv_fp):
            continue
        ensemble_df = pd.read_csv(csv_fp, index_col="model_id")
        ensemble_df.drop("pdb_fp", axis=1, inplace=True)
        ensemble_df["pdb_id"] = pdb_id
        ensemble_df["sampling_method"] = sampling_method
        if df is None:
            df = ensemble_df
        else:
            df = pd.concat([df, ensemble_df])

df

In [ ]:
pdb_ids = sorted(df.pdb_id.unique())
print(len(pdb_ids))

In [ ]:
results = dict()

for pdb_id in tqdm(pdb_ids):
    results[pdb_id] = dict()
    template = TCRpMHCII(fp=f"../data/mhcii_tcr_templates/{pdb_id}.pdb", id=f"{pdb_id}__crystal")
    scorer = DockQScorer(template, ["P", ("A", "B")])
    af3_model = TCRpMHCII(fp=f"../data/mhcii_tcr_models/alphafold3/{pdb_id}.pdb", id=f"{pdb_id}__af3")
    sub_df = df[df.pdb_id == pdb_id]
    for sampling_method in df.sampling_method.unique():
        sub_sub_df = sub_df[sub_df.sampling_method == sampling_method]
        results[pdb_id][sampling_method] = sub_sub_df.dockq.values.max()
    results[pdb_id]["alphafold3"] = scorer(af3_model)
    try:
        tfold_model = TCRpMHCII(fp=f"../data/mhcii_tcr_models/tfold-tcr/{pdb_id}.pdb", id=f"{pdb_id}__tfold")
        results[pdb_id]["tfold_tcr"] = scorer(tfold_model)
    except:
        results[pdb_id]["tfold_tcr"] = 0

results_df = pd.DataFrame.from_dict(results, orient="index")

In [ ]:
import numpy as np

matrices = list()
traces = dict()
for z in sampling_methods + ["tfold_tcr", "alphafold3"]:
    traces[z] = list()

df = df[df.sampling_method != "ensemble/pandora2"]

for pdb_id in pdb_ids:

    for z in ["tfold_tcr", "alphafold3"]:
        traces[z].append(results[pdb_id][z])

    matrix = np.zeros((len(df.sampling_method.unique()), len(results_df.columns)))

    for i, x in enumerate(df.sampling_method.unique()):
        scores = df[(df.pdb_id == pdb_id) & (df.sampling_method == x)].dockq
        traces[x].append(scores.max())
        for j, y in enumerate(results_df.columns):
            matrix[i, j] = int(scores.max() > results_df.loc[pdb_id][y])

    matrices.append(matrix)

import plotly.express as px
import plotly.graph_objects as go

fig = go.Figure(data=go.Heatmap(z=np.array(matrices).mean(axis=0) * 100, x=results_df.columns, y=df.sampling_method.unique(), texttemplate="%{z:.1f}%"))
fig.update_yaxes(autorange="reversed")
fig.show()

fig = go.Figure()
for t, y in traces.items():
    fig.add_trace(go.Scatter(x=pdb_ids, y=y, mode="markers", name=t))
fig.update_layout(plot_bgcolor="white")
fig.show()